# Análise exploratória: PCA e MDS com dados da ANP

## 1. Introdução do problema

O trabalho parte de uma pergunta simples para quem opera no varejo de combustível: diferentes estados e meses se comportam de modo parecido em **preço**, **volume**, **mistura entre gasolina e etanol** e **oscilações mês a mês**, ou dá para enxergar grupos bem distintos? Isso importa quando se pensa em estoque, mix de produto e estratégia regional — antes de partir para modelos mais pesados, faz sentido mapear padrões e exceções.

Usamos **PCA** para condensar a variabilidade das variáveis numéricas em duas direções principais e enxergar cargas — ou seja, quais variáveis puxam mais cada eixo — e **MDS** para projetar registros próximos ou afastados com base nas distâncias no espaço padronizado. Os dois métodos falam de **associação e semelhança**, não de causa e efeito: distâncias e componentes ajudam a formar hipóteses, não provam que um preço “gerou” tal volume.

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path("..").resolve()))

import matplotlib.pyplot as plt
import pandas as pd
from src.data_preparation import FEATURES_NUMERICAS, padronizar_features, preparar_dados
from src.mds_analysis import aplicar_mds
from src.pca_analysis import aplicar_pca

## 2. Descrição das bases

Foram combinadas duas fontes públicas da ANP:

- **Preços médios de revenda** por UF e mês (gasolina C e etanol hidratado), arquivo de série mensal estadual disponível na página da ANP.
- **Vendas em volume** dos mesmos combustíveis, por UF e mês, no conjunto `vendas-combustiveis-m3`.

O período trabalhado no notebook é configurável nas funções; no código abaixo usamos 2021 a 2025, alinhados ao relatório gerado pelo script `run_analysis.py`. Cada linha final é um **par UF × mês** com preço e volume coerentes após o cruzamento das tabelas.

## 3. Preparação dos dados

O fluxo de limpeza: normalizar grafias, filtrar gasolina C e etanol hidratado, pivotar preços e vendas por `mes_ano` e UF, e fazer o **merge interno só com `mes_ano` + sigla `uf`** (o nome completo do estado entra depois com `combine_first`). Em seguida derivamos participação do etanol, razão de volumes, preço relativo e variações percentuais **por UF**.

Valores ausentes ou infinitos nas features numéricas são descartados ao final dessa montagem para não contaminar PCA e MDS.

In [ ]:
dados = preparar_dados(periodo_inicio=2021, periodo_fim=2025)
dados.shape

## 4. Correção do problema no merge

Um ponto onde o projeto já tropeçou antes: as duas bases às vezes rotulam a **região** de forma inconsistente (“Centro Oeste” contra “Centro-Oeste”, grafias diferentes após padronização, etc.). Se a mescla usasse só uma coluna dessas como chave, algumas UF sumiriam só por causa de inconsistência textual, não por falta de dado.

O problema clássico era **Centro-Oeste** nas duas fontes: `Centro Oeste` com espaço, `Centro-Oeste` com hífen, outras variações após normalizar acentos. Se o `merge` exige `uf_nome` idêntico, dá **join vazio** para DF, GO, MS e MT e são dezenas de meses vezes quatro UFs — da ordem de **~240 linhas** que sumiam sem critério de negócio. A correção foi cruzar **apenas mês e sigla `uf`**, tratar `uf_nome` e `região` como colunas reconciliáveis depois do join e manter `normalizar_regiao` para alinhar o rótulo regional.

## 5. Features analisadas

### Seleção das features

O conjunto não foi montado aleatoriamente; cada grupo responde a um recorte diferente da leitura de mercado. Preços e preço relativo ajudam a comparar competitividade regional; volumes em m³ dizem algo sobre escala absoluta da demanda; participação do etanol e a razão etanol/gasolina descrevem o papel do biocombustível **no mix** observado nas vendas (com ressalvas que comentamos adiante); as variações mensais pegam sobretudo **dinâmica recente**, não só o patamar médio entre UFs.

`participação_etanol` e `razao_etanol_gasolina` contam histórias parecidas — ambas orbitam quanto espço o etanol ocupa ao lado da gasolina — mas não são redundantemente iguais: uma é proporção dentro da soma dos dois combustíveis; outra é a razão direta entre volumes de etanol e de gasolina. Ainda assim, vale ter em mente que o PCA pode acabar dando peso forte a uma mesma dimensão “etanol versus gasolina” por causa dessa proximidade semântica; é limitação transparente da escolha, não necessariamente um erro nos dados.

| Dimensão | Features | Justificativa |
|---|---|---|
| Preço | `preco_medio_gasolina_c`, `preco_relativo_etanol_gasolina`, `variacao_preco_gasolina_c` | Patamar do custo da gasolina ao consumidor e posição relativa do etanol; a variação mensal marca choques ao longo do tempo. |
| Volume de mercado | `volume_gasolina_c_m3`, `volume_etanol_hidratado_m3` | Escalas absolutas da demanda — úteis, mas também arrastam estados muito grandes; voltamos nisso depois dos outliers. |
| Substituição entre combustíveis | `participacao_etanol`, `razao_etanol_gasolina` | Quanto o etanol “pesa” no consumo quando olhamos gasolina versus etanol. |
| Dinâmica mensal | `variacao_volume_gasolina_c` | Oscilações mês a mês nas vendas de gasolina dentro de cada UF. |

Lista canônica usada no projeto (mantida igual ao código em `src/data_preparation.py`):

In [ ]:
FEATURES_NUMERICAS

Amostra do painel já **antes** da padronização:

In [ ]:
dados[
    [
        "mes_ano",
        "uf",
        "regiao",
        "preco_medio_gasolina_c",
        "volume_gasolina_c_m3",
        "volume_etanol_hidratado_m3",
        "participacao_etanol",
        "variacao_volume_gasolina_c",
    ]
].head()

## 6. Padronização

As colunas não nasceram comparáveis: preços em reais, volumes em metros cúbicos e participações em proporção. Para os algoritmos isso faz diferença: distâncias e componentes ficam mais sensíveis aos números com maior dispersão absoluta antes de qualquer reflexão estatística sobre “quem importa”.

Aplicamos **`StandardScaler`**, que empurra cada feature para média próxima de zero e desvio um. Isso equilibra magnitudes quando calculamos PCA e distâncias euclidianas para o MDS. Não finge remover o papel de grandes mercados no perfil multimodal dos pontos — estados gigantes continuam poderendo aparecer extremos combinando várias variáveis — mas evita que uma única grandeza física dicte só pelo tamanho bruto das unidades escolares.

In [ ]:
dados_padronizados, _scaler = padronizar_features(dados)
dados_padronizados.describe().round(3)

## 7. PCA — aplicação

### Como estamos usando o método

O PCA lineariza uma compressão das oito features em dois eixos ortogonais. O primeiro eixo maximiza variância; o segundo pega boa parte do que sobrou sendo forçado a não repetir direção da primeira.

Na prática exploratória foi usado para pintar registros UF-mês no plano, medir parcela da variância explicada (algo que **o MDS não oferece do mesmo modo**) e inspecionar **cargas** — sempre lembrando que reduzir de oito para duas dimensões joga informação fora. O número exato de percentuais aparece logo abaixo na tabela; trate o gráfico biplot mentalmente como retrato sintético, não como foto completa do dataset.

In [ ]:
pca_df, cargas, variancia, _pca_full = aplicar_pca(dados_padronizados)
display(variancia)
display(cargas.sort_values("PC1", key=abs, ascending=False))

**Gráfico de cargas** — barras horizontalizadas para ler o peso direto de cada variável em PC1 e PC2 (coeficientes nos componentes, coerentes com a tabela acima).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, col, titulo in (
    (axes[0], "PC1", "Cargas — componente 1"),
    (axes[1], "PC2", "Cargas — componente 2"),
):
    s = cargas[col].sort_values(key=abs)
    ax.barh(s.index.astype(str), s.values, color="steelblue")
    ax.axvline(0.0, color="black", linewidth=0.7)
    ax.set_title(titulo)
plt.tight_layout()
plt.show()

## 8. Interpretação das cargas do PCA

Cada carga diz **quanto** daquele componente aquela variável empurra, com sinal positivo ou negativo; não é causalidade nem importância preditiva fora dessa ortogonalização específica.

No **primeiro componente** aparecem, com peso alto, `participacao_etanol`, `razao_etanol_gasolina`, volumes de etanol e de gasolina e `preco_relativo_etanol_gasolina` — um bloco coerente com **perfil de mix** e com **posição de preço do etanol** frente à gasolina. É o eixo em que estados com etanol muito presente no consumo tendem a se separar dos de participação baixa, embora volumes absolutos também entrem na conta (conforme a seção sobre limitação).

No **segundo componente** o quadro é outro: quem mais puxa é o trio **variação mensal de preço da gasolina**, **volume corrente de gasolina** (em geral com sinal oposto ao das taxas de variação — confira na tabela ordenada pelo PC2) e **variacao_volume_gasolina_c**, seguido de `preco_medio_gasolina_c` e ainda alguma pressão do preço relativo etanol/gasolina. Em síntese, PC2 mistura **choques ou oscilações mês a mês** com **patamar de preço da gasolina e de volume vendido**, ou seja, dinâmicas temporárias combinadas ao que não ficou inteiramente engolido pelo primeiro eixo.

**Cuidados:** isso sintetiza associações lineares combinadas pelo PCA na amostra; não autoriza inferir política tributária, culpar preço específico de um resultado de mercado nem ignorar correlações preexistentes entre variáveis (como já comentamos para participação versus razão do etanol).

### Projeção no plano (PC1 × PC2)

Cada bolha ou ponto espelha um mês dentro de uma UF. A intensidade pela participação do etanol só é codificação de cor para leitura visual — outros esquemas de cor por região também são válidos se quiser destacar geopolítico em vez de mix.

In [ ]:
pca_plot = dados.join(pca_df)
ax = pca_plot.plot.scatter(
    x="PC1",
    y="PC2",
    c="participacao_etanol",
    colormap="viridis",
    figsize=(8, 6),
)
ax.set_title("PCA — participação do etanol (cor)")
ax.set_xlabel(f"PC1 ({variancia.loc[0, 'variancia_explicada']:.1%})")
ax.set_ylabel(f"PC2 ({variancia.loc[1, 'variancia_explicada']:.1%})")
plt.show()

## 9. Outliers na projeção PCA

Uma medida rápida: distância euclidiana da origem no plano PC1–PC2 (outras definições de outlier existiriam; esta é transparente e fácil de reproduzir). Registros no topo da lista ficaram **mais extremos nesse resumo bidimensional** — em geral combinação de escala, mix e/ou mês atípico.

In [ ]:
pca_plot = dados.join(pca_df)
pca_plot["distancia_pca"] = (pca_plot["PC1"] ** 2 + pca_plot["PC2"] ** 2) ** 0.5
cols = [
    "mes_ano",
    "uf",
    "regiao",
    "preco_medio_gasolina_c",
    "volume_gasolina_c_m3",
    "participacao_etanol",
    "variacao_volume_gasolina_c",
    "PC1",
    "PC2",
    "distancia_pca",
]
pca_plot.sort_values("distancia_pca", ascending=False)[cols].head(12)

### Limitação: volumes absolutos

Com volumes em nível cru (m³), estados cuja frota/população já consome muito aparecem com mais facilidade na cauda da distribuição de distâncias e nas bordas do PCA — **SP costuma aparecer porque realmente está em escala incomum**, não porque o método tenha decidido penalizar paulistas. O ponto é outro: parte do que lemos como “longe no gráfico” pode ser **efeito de porte de mercado** misturado com diferenças de **perfil** (quanto etanol entra no mix, etc.).

Próximos passos naturais seriam refazer o exercício com transformações **logarítmicas nos volumes**, ou definir algo como **consumo por habitante** usando população por UF como denominador externo à ANP, e comparar como mudam outliers e cargas.

## 10. MDS — aplicação

### Leitura do método

O MDS aparece aqui como **contraste de leitura**: em vez de buscar eixos que maximizam variância, ele tenta colocar pontos em 2D de forma que distâncias no plano imitem distâncias no espaço original das oito features padronizadas. Por isso a interpretação central é **quem está perto de quem**; eixos `MDS1` e `MDS2` não vêm com rótulo automático de “preço” ou “etanol”.

Isso é força e fraqueza: bom para ver **agrupamentos e vizinhanças**, ruim se a pergunta for “qual variável move o eixo horizontal”. Subamostra de até 800 pontos (como no `run_analysis.py`) mantém o gráfico leve; o valor de stress impresso no notebook é referência numérica do ajuste, útil comparando tentativas, não um “R²”.

In [ ]:
mds_df, mds = aplicar_mds(dados_padronizados, max_registros=800)
print(f"Stress MDS: {mds.stress_:.2f}")
mds_df.head()

### Gráfico do MDS

Novamente a cor segue participação do etanol só para dar contexto visual; a mensagem principal é **distância relativa** entre pontos, não o valor absoluto dos eixos.

In [ ]:
mds_plot = dados.loc[mds_df.index].join(mds_df)
ax = mds_plot.plot.scatter(
    x="MDS1",
    y="MDS2",
    c="participacao_etanol",
    colormap="plasma",
    figsize=(8, 6),
)
ax.set_title("MDS — proximidade entre registros (cor = participação etanol)")
ax.set_xlabel("MDS1")
ax.set_ylabel("MDS2")
plt.show()

## 11. Comparação entre PCA e MDS

O **PCA** entrega caminho explícito do que está sendo comprimido: variância explicada, cargas, leitura direcional de variáveis. Útil para defender em apresentação *por que* dois estados caem em lados opostos se as cargas mostram que mix de etanol e escala puxam o primeiro eixo.

O **MDS** economiza essa ponte variável a variável, mas devolve mapa onde **semelhança global** importa mais: clusters coerentes com o PCA costumam aparecer, porém com distorções diferentes nas bordas. Em suma: PCA para **estrutura explicável linearmente**, MDS para **vizinhança** sem se comprometer com componentes de variância.

## 12. Limitações

- **Nada aqui é causal**: proximidade ou carga alta não prova que preço “causou” volume; no máximo descreve padrão conjunto.
- **Duas dimensões** perdem informação; decisões operacionais maiores deveriam checar sensibilidade com mais componentes ou com variáveis alternativas.
- **Correlação entre features** (especialmente participação e razão etanol/gasolina) pode inflar a “importância” aparente de uma narrativa de mix — às vezes vale rodar versão do PCA sem uma delas para ver o que muda.
- **Volumes absolutos** misturam escala e perfil; ver seção dedicada acima.
- **Dados agregados da ANP** não trazem renda, frota detalhada, eventos locais, etc.; interpretação fica no nível macro mensal por UF.

## 13. Conclusão

PCA e MDS juntos deram um mapa razoável de como preço, escala de vendas, presença do etanol e oscilações mês a mês se combinem para separar ou aproximar estados e meses nesse recorte da ANP. O PCA foi o instrumento para amarrar **quais variáveis puxam** cada direção resumida; o MDS complementou com **quem fica ao lado de quem** no espaço das distâncias.

As limitações — volumes absolutos, par de variáveis quase contando a mesma história de mix, janela curta se comparada a décadas inteiras de mercado — não anulam o exercício: só deixam claro o que seria natural evoluir depois (log-volume, per capita, poda de feature sensível, checagem de robustez). Para o objetivo exploratório do projeto, o ganho principal é ter uma base visual e numérica comum para discutir regionalização e meses fora do padrão antes de modelos ou decisões mais caras.